# PhoBERT ABSA — SOTA Preprocessing Experiment

**Branch**: `feature/sota-preprocessing`  
**Thay đổi so với baseline**: thêm Vietnamese error correction (`bmd1905/vietnamese-correction-v2`)  
**Encoder**: `cls_only` (kết quả ablation tốt nhất)  
**Mục tiêu**: kiểm tra liệu error correction có cải thiện Combined F1 không

| Config | Giá trị |
|--------|--------|
| Model | `vinai/phobert-base-v2` |
| Encoder | `cls_only` (768-dim CLS) |
| Loss | Weighted Focal Loss (weight_clip=10.0) |
| Preprocessing | Unicode + Teencode + **Error Correction** + VnCoreNLP |
| Baseline Combined F1 | 0.5543 |
| SOTA Combined F1 | 0.7732 |

In [ ]:
# Cell 1 — Check GPU & install dependencies
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu}')
    print(f'  VRAM: {vram:.1f} GB')
    if vram < 14:
        print('  [WARN] VRAM < 14GB — có thể cần giảm batch_size')
else:
    raise RuntimeError('Không có GPU\!')

torch.cuda.empty_cache()
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

# Install dependencies
# bmd1905/vietnamese-correction-v2 là seq2seq model, cần sentencepiece
\!pip install -q transformers==4.38.0 underthesea py_vncorenlp tabulate tqdm \
             scikit-learn sentencepiece sacremoses
print('Dependencies installed')

In [ ]:
# Cell 2 — Clone repo từ branch feature/sota-preprocessing
import os, sys

REPO_URL    = 'https://github.com/vudinhminh08/NLP-project-master-study.git'
REPO_BRANCH = 'feature/sota-preprocessing'
PROJECT_DIR = '/kaggle/working/absa-project'

if not os.path.exists(PROJECT_DIR):
    print(f'Cloning {REPO_URL} (branch: {REPO_BRANCH})...')
    \!git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
    print('Clone completed')
else:
    print(f'Repo đã tồn tại — pulling latest ({REPO_BRANCH})...')
    \!cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')
print(f'Branch: ' + str(\!git rev-parse --abbrev-ref HEAD).strip())

for d in ['data', 'outputs/models_cls_only_sota', 'outputs/results_cls_only_sota', 'outputs/eda']:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, 'code/data_processing')
sys.path.insert(0, 'code/phobert')

print('\nCấu trúc repo:')
\!ls -la
\!ls code/data_processing/ code/phobert/

In [ ]:
# Cell 3 — Download VLSP 2018 Hotel dataset
import pandas as pd, os

if not os.path.exists('data/train.csv'):
    print('Downloading VLSP 2018 Hotel dataset...')
    \!git clone https://github.com/ds4v/absa-vlsp-2018.git /tmp/ds4v --depth=1
    \!cp /tmp/ds4v/datasets/vlsp2018_hotel/train.csv data/
    \!cp /tmp/ds4v/datasets/vlsp2018_hotel/dev.csv data/
    \!cp /tmp/ds4v/datasets/vlsp2018_hotel/test.csv data/
    print('Data downloaded')
else:
    print('Data already available')

for split in ['train', 'dev', 'test']:
    df = pd.read_csv(f'data/{split}.csv')
    print(f'  {split}: {len(df)} rows × {df.shape[1]} cols')

In [ ]:
# Cell 4 — SOTA Preprocessing
# Pipeline: Unicode → Teencode → Error Correction → VnCoreNLP word segmentation
# Tạo ra data/*_preprocessed_sota.csv (giữ nguyên *_preprocessed.csv baseline)
import pandas as pd, os

FORCE_REPROCESS = False  # Đặt True để chạy lại dù cache đã có

sota_files_exist = all(
    os.path.exists(f'data/{s}_preprocessed_sota.csv')
    for s in ['train', 'dev', 'test']
)

if (not FORCE_REPROCESS) and sota_files_exist:
    print('Cache SOTA available (data/*_preprocessed_sota.csv)')
    s = pd.read_csv('data/train_preprocessed_sota.csv').iloc[0]
    print(f'  Original : {s["Review"][:80]}')
    print(f'  Processed: {str(s.get("processed_review", "N/A"))[:80]}')
else:
    print('Chạy SOTA preprocessing (có thể mất 15-30 phút trên T4)...')
    from step3_preprocessing import (
        preprocess_dataframe, VnCoreNLPSegmenter, VietnameseErrorCorrector
    )
    import py_vncorenlp

    # Setup VnCoreNLP
    vncorenlp_dir = os.path.join(os.getcwd(), 'vncorenlp')
    if not os.path.exists(os.path.join(vncorenlp_dir, 'models',
                                        'wordsegmenter', 'wordsegmenter.rdr')):
        print('Downloading VnCoreNLP models...')
        py_vncorenlp.download_model(save_dir=vncorenlp_dir)
    segmenter = VnCoreNLPSegmenter(vncorenlp_dir=vncorenlp_dir, use_fallback=False)

    # Setup Error Corrector
    corrector = VietnameseErrorCorrector(
        model_name='bmd1905/vietnamese-correction-v2',
        device='cuda',
        batch_size=32,
        use_fallback=True,
    )

    for split in ['train', 'dev', 'test']:
        df = pd.read_csv(f'data/{split}.csv')
        print(f'\n--- {split} ({len(df)} samples) ---')
        df = preprocess_dataframe(
            df,
            segmenter=segmenter,
            corrector=corrector,
            cache_path=f'data/{split}_preprocessed_sota.csv',
        )
        print(f'  Done: {len(df)} samples')
        print(f'  Example: {df.iloc[0]["processed_review"][:100]}')

    segmenter.close()
    print('\nSOTA Preprocessing hoàn tất\!')

# So sánh baseline vs SOTA preprocessing
if os.path.exists('data/train_preprocessed.csv'):
    base = pd.read_csv('data/train_preprocessed.csv').iloc[0]
    sota = pd.read_csv('data/train_preprocessed_sota.csv').iloc[0]
    print('\n=== So sánh preprocessing ===')
    print(f'Original : {base["Review"][:100]}')
    print(f'Baseline : {str(base.get("processed_review", "N/A"))[:100]}')
    print(f'SOTA     : {str(sota.get("processed_review", "N/A"))[:100]}')

In [ ]:
# Cell 5 — Load EDA config và class weights
# (chạy step1_eda.py nếu chưa có outputs/eda/)
import json, os

if not os.path.exists('outputs/eda/encoder_config.json'):
    print('Chạy EDA để tạo encoder_config.json...')
    \!python code/data_processing/step1_eda.py

# Ghi đè encoder_option = cls_only
enc_cfg_path = 'outputs/eda/encoder_config.json'
enc_cfg = json.load(open(enc_cfg_path))
enc_cfg['encoder_option'] = 'cls_only'
with open(enc_cfg_path, 'w') as f:
    json.dump(enc_cfg, f, indent=2)
print('=== Encoder Config ===')
for k, v in enc_cfg.items():
    print(f'  {k}: {v}')

cw = json.load(open('outputs/eda/class_weights.json'))
print('\n=== Global Class Weights ===')
label_map = {'0': 'absent', '1': 'positive', '2': 'negative', '3': 'neutral'}
for cls, w in cw['global_weights'].items():
    print(f'  {label_map.get(cls,cls):12s}: {float(w):.1f}x')

from utils.constants import TRAIN_CONFIG, PHOBERT_MODEL_NAME
print('\n=== Train Config ===')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')
print(f'  model: {PHOBERT_MODEL_NAME}')
assert PHOBERT_MODEL_NAME == 'vinai/phobert-base-v2'
print('\nConfig OK')

In [ ]:
# Cell 6 — Training PhoBERT cls_only + SOTA preprocessing
# Output: outputs/models_cls_only_sota/ + outputs/results_cls_only_sota/
import torch
torch.cuda.empty_cache()
print(f'VRAM free before training: '
      f'{(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9:.1f} GB')

from run_experiment import main

test_metrics = main(
    encoder_option='cls_only',
    use_amp=True,
    data_suffix='_sota',   # dùng *_preprocessed_sota.csv
)

print('\n' + '='*60)
print('KẾT QUẢ — cls_only + SOTA preprocessing')
print(f'  ACD F1:      {test_metrics["macro_acd_f1"]:.4f}  (SOTA ref: 0.8255)')
print(f'  SPC F1:      {test_metrics["macro_spc_f1"]:.4f}')
print(f'  Combined F1: {test_metrics["macro_combined_f1"]:.4f}  (SOTA ref: 0.7732)')
print(f'  Baseline:    0.5543  (cls_only, không có error correction)')
delta = test_metrics['macro_combined_f1'] - 0.5543
print(f'  Delta vs baseline: {delta:+.4f} ({delta*100:+.2f}%)')
print('='*60)

In [ ]:
# Cell 7 — So sánh SOTA preprocessing vs Baseline
import json, os

results = {}

# SOTA results (vừa train xong)
sota_path = 'outputs/results_cls_only_sota/phobert_test_metrics.json'
if os.path.exists(sota_path):
    results['cls_only + SOTA preprocessing'] = json.load(open(sota_path))

# Baseline results (nếu đã có từ lần train trước)
base_path = 'outputs/results_cls_only/phobert_test_metrics.json'
if os.path.exists(base_path):
    results['cls_only + Baseline preprocessing'] = json.load(open(base_path))

print('=== So sánh kết quả ===')
print(f'{"":45s} {"ACD F1":>8} {"SPC F1":>8} {"Comb F1":>9}')
print('-' * 75)
for name, m in results.items():
    print(f'{name:45s} {m["macro_acd_f1"]:>8.4f} {m["macro_spc_f1"]:>8.4f} {m["macro_combined_f1"]:>9.4f}')
print(f'{"SOTA (ds4v, concat_4_layers + complex preproc)":45s} {0.8255:>8.4f} {"—":>8} {0.7732:>9.4f}')

In [ ]:
# Cell 8 — Learning curve
import json, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

history_path = 'outputs/results_cls_only_sota/training_history.json'
if not os.path.exists(history_path):
    print('Chưa có training history')
else:
    history = json.load(open(history_path))
    best_ep = history['best_epoch']
    epochs  = range(1, len(history['train_loss']) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('PhoBERT cls_only + SOTA Preprocessing — Learning Curve', fontsize=13)

    ax1.plot(epochs, history['train_loss'], 'o-', c='crimson',   lw=2, label='Train Loss')
    ax1.plot(epochs, history['dev_loss'],   'o-', c='steelblue', lw=2, label='Dev Loss')
    ax1.axvline(best_ep, c='green', ls='--', alpha=0.7, label=f'Best (epoch {best_ep})')
    ax1.set(title='Loss', xlabel='Epoch', ylabel='Cross-Entropy Loss')
    ax1.legend(); ax1.grid(alpha=0.3)

    ax2.plot(epochs, history['dev_acd_f1'],      's-', c='darkorange', lw=2, label='Dev ACD F1')
    ax2.plot(epochs, history['dev_spc_f1'],      '^-', c='purple',     lw=2, label='Dev SPC F1')
    ax2.plot(epochs, history['dev_combined_f1'], 'o-', c='green',      lw=2.5, label='Dev Combined F1')
    ax2.axvline(best_ep, c='green', ls='--', alpha=0.7, label=f'Best (epoch {best_ep})')
    ax2.axhline(0.7732, c='red', ls=':', alpha=0.5, label='SOTA 0.7732')
    ax2.axhline(0.5543, c='gray', ls=':', alpha=0.5, label='Baseline 0.5543')
    ax2.set(title='F1 Score', xlabel='Epoch', ylabel='Macro F1')
    ax2.legend(); ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('outputs/eda/learning_curve_sota.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Best epoch: {best_ep} | Best Combined F1: {history["best_combined_f1"]:.4f}')
    print('Saved: outputs/eda/learning_curve_sota.png')

In [ ]:
# Cell 9 — Download kết quả
import os, json, shutil

report = 'outputs/results_cls_only_sota/phobert_summary.md'
if os.path.exists(report):
    print(open(report, encoding='utf-8').read())
else:
    print('Chưa có report')

# Zip toàn bộ results
shutil.make_archive('/kaggle/working/phobert_sota_results', 'zip', 'outputs')
print('Zip: /kaggle/working/phobert_sota_results.zip')
print('   → Kaggle Output panel → Download')

# Tổng hợp
print('\n=== Kết quả cuối ===')
m_path = 'outputs/results_cls_only_sota/phobert_test_metrics.json'
if os.path.exists(m_path):
    m = json.load(open(m_path))
    print(f'SOTA_PREPROCESSING_ACD_F1      = {m["macro_acd_f1"]:.4f}')
    print(f'SOTA_PREPROCESSING_SPC_F1      = {m["macro_spc_f1"]:.4f}')
    print(f'SOTA_PREPROCESSING_COMBINED_F1 = {m["macro_combined_f1"]:.4f}')
    print(f'DELTA_VS_BASELINE              = {m["macro_combined_f1"] - 0.5543:+.4f}')